# Which way did the ground first move, and can a machine tell?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FAI4EPS%2FEPS88_PyEarth&branch=main&urlpath=lab%2Ftree%2FEPS88_PyEarth%2Fdocs/notebooks%2FT8_which_way_first.ipynb).

When a fault slips, the very first thing it does to the ground at any one station is either push
it up or pull it down — and which of the two happens depends on where that station sits relative
to the fault. Read the first motion at enough stations and you can work backwards to the
orientation of the fault that broke and the direction it slipped, which is how almost every
earthquake mechanism in the record was determined before computers were involved. It is one
letter of information per recording: **U** or **D**.

An analyst reads that letter by eye, in a fraction of a second, from the first wiggle after the P
arrives. This notebook has 2,348 of their readings — 1,233 up and
1,115 down — on recordings of 866 Northern California earthquakes, and the
question is whether a machine can do the same job. The interesting part is not whether it can. It
is how little of the seismogram it needs, and what it is doing with the rest.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by an empty cell. Fill them all in, run every cell so your answers and figures are
saved in the file, then **download this notebook itself — the `.ipynb` file — and upload it
to Gradescope.** Not a PDF: the marking reads your notebook, and a PDF cannot be read.
In JupyterLab: **File ▸ Download**, or right-click the file in the left-hand panel and choose
**Download**.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## How this notebook is different

This is a **project track**. It is not a weekly notebook and it does not behave like one.

A weekly notebook shows you a move, walks you through it, and then asks you to make it once
yourself. This one loads the data and reproduces the single result its title rests on — that one
line of arithmetic can read a first motion — and then stops helping. From there on every section
is a sentence describing what to find out and an empty cell to find it out in. There is no worked
example above to pattern-match against, because on a real question there never is one.

**There is exactly one self-check in this notebook, and it is on the data loading.** After that,
nothing tells you whether you are right. That is not an oversight and it is not laziness: past
the loading step there is no single right answer here, so a cell that said `assert` would be
lying to you about how research works. What replaces it is the thing researchers actually use — a
number you can get two ways, a result you can predict before you compute it, and a claim you can
try to break.

**And it does not close.** The last section is a question this course does not know the answer
to. Everything above it is scaffolding; that question is the project.

## What you'll be able to do

**The science.** Say how much of a seismogram the polarity of the first motion actually lives in,
and defend the answer with a number rather than an adjective. Then say whether a trained network
is worth its cost on this problem, and what is stopping every method from doing better.

**The skills.** Split data by the structure that is really in it rather than by row. Write down
the dumbest possible method first and make everything else beat it. Train a small
one-dimensional convolutional network on a signal, and change one thing about its input at a time
until the change tells you something.

**The four questions, in order:**

1. Which way did the ground first move, and can one line of arithmetic tell?
2. How much waveform does the network need?
3. Did the network earn its place?
4. Is what is left the model, or the labels?

The open question at the end is not on that list. It is the project; the four above are what you
build to reach it.

## Setup

The waveforms are 44 MB — far too big to keep beside the notebook, so
they arrive from a release of the course repository the first time you run the cell below and are
kept on disk after that.

Three things about the file are worth reading before you start:

- `polarity` holds one character per recording: `"U"`, `"D"`, or nothing at all.
  152 recordings carry no polarity, and the only safe filter is the positive one
  — keep the rows that say `U` or `D`, rather than dropping the rows that look empty.
- Each waveform row has been divided by its own typical size, so a `1` means *one typical wiggle
  for this instrument on this recording*, not a fixed number of nanometres. The file is also
  already cut off at ±10: 6.2% of all its samples sit exactly on
  that bound, so the tops of the largest arrivals have been shaved off. Any *size* you measure
  therefore has our ceiling on it — and it is worth holding on to the thought that a cut-off can
  do more to a recording than shave the top off a peak.
- Polarity is read on the **up-down** component, which is row 2 of the three. Rows 0 and 1 are
  the two horizontal directions, and the first motion is not defined on them.

The file also carries `magnitude`, `distance_km`, `depth_km`, `station` and `snr`, one per
recording. Any of them comes out the same way as the arrays below, filtered with the same mask.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.metrics import accuracy_score

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

WAVEFORMS = ("https://github.com/AI4EPS/EPS88_PyEarth/releases/download/data-v1/phasenet_ncedc.npz")


def load():
    """Read the waveform file, downloading it from the course release the first time."""
    try:
        return np.load("phasenet_ncedc.npz")
    except FileNotFoundError:
        torch.hub.download_url_to_file(WAVEFORMS, "phasenet_ncedc.npz", progress=False)
        return np.load("phasenet_ncedc.npz")


data = load()
labelled = (data["polarity"] == "U") | (data["polarity"] == "D")

vertical = data["waveform"][labelled][:, 2, :]   # the up-down component, one row per recording
p_index = data["p_index"][labelled].astype(int)  # sample number of the analyst's P pick
up = data["polarity"][labelled] == "U"           # True where the analyst wrote U
event_id = data["event_id"][labelled]            # which earthquake each recording is of
snr = data["snr"][labelled]                      # how much louder the quake is than the background
SAMPLE_RATE = 100                             # samples per second
LEAD_IN = 10                                   # samples of background kept before every pick

print("recordings with a polarity:", len(vertical), "of", len(data["polarity"]))
print("first motion up:", up.sum(), " down:", (~up).sum())
print("earthquakes:", len(np.unique(event_id)), " stations:", len(np.unique(data["station"][labelled])))
print("each recording:", vertical.shape[1], "samples =",
      round(vertical.shape[1] / SAMPLE_RATE, 2), "seconds")
print("the P arrives between", round(p_index.min() / SAMPLE_RATE, 2), "and",
      round(p_index.max() / SAMPLE_RATE, 2), "seconds in")
print("largest value anywhere in the file:", round(float(np.abs(data["waveform"]).max()), 2))

In [ ]:
assert vertical.shape == (len(up), 2048), \
    "the waveforms are not the shape this notebook expects — the file was read wrong"
assert 1000 < up.sum() < 1400 and 1000 < (~up).sum() < 1400, \
    "the two polarities should be close to balanced; they are not, so the filter is wrong"
assert (p_index > LEAD_IN).all() and (p_index + 1000 < vertical.shape[1]).all(), \
    "every pick must leave room for the windows below — some do not"
print(f"✓ the data — {len(vertical)} recordings carrying a polarity, {up.sum()} up and "
      f"{(~up).sum()} down, from {len(np.unique(event_id))} earthquakes")

### And that is the last self-check in this notebook

The pipeline is now trustworthy: the file is the file, the filter is the filter, the numbers
below are the numbers. Everything from here is yours, and nothing will tell you when you have it
right.

## Which way did the ground first move, and can one line of arithmetic tell?

The P wave is the compression that arrives first. If the fault's motion pushed the rock towards
this station, the ground's first move is **away from the source** — upwards at the surface — and
the analyst writes `U`. If it pulled, the first move is downwards, and they write `D`. Everything
after that first swing is the rest of the earthquake: more P energy, the S wave, reflections, the
ground ringing. None of it is the first motion.

So the whole of the label ought to live in the handful of samples immediately after the pick. The
figure below is two recordings, one of each letter, and it is worth looking at before anything is
computed.

In [ ]:
def cut(seconds):
    """Every recording's vertical trace, from just before the pick to `seconds` after it."""
    n = int(seconds * SAMPLE_RATE)
    out = np.zeros((len(p_index), LEAD_IN + n), dtype="float32")
    for i in range(len(p_index)):
        out[i] = vertical[i, p_index[i] - LEAD_IN:p_index[i] + n]
    return out

In [ ]:
half_second = cut(0.5)
time = (np.arange(half_second.shape[1]) - LEAD_IN) / SAMPLE_RATE

plt.plot(time, half_second[1819], color="0.2", label="the analyst wrote U")
plt.plot(time, half_second[731], color="firebrick", label="the analyst wrote D")
plt.axvline(0, color="steelblue", lw=1.2)
plt.xlabel("seconds from the analyst's P pick")
plt.ylabel("ground motion (in units of this trace's own background)")
plt.title("Two first motions of the 2,348; the line is the pick")
plt.legend()
plt.show()

Both traces sit on nothing until the blue line and then leave it, one upwards and one downwards,
within a few hundredths of a second. That is the entire signal this project is about.

Before measuring anything, the data has to be cut in two, and **how** it is cut is the first
place this problem can be got wrong. The 2,348 recordings are of only
866 earthquakes, so the same earthquake is recorded at many stations at once. Split
the rows at random and most earthquakes end up on both sides of the wall — the model sees a
recording of an earthquake, then is scored on another recording of the same one. Splitting by
earthquake is the honest cut here.

**Train/test split:** Hide some data from yourself, then check.

In [ ]:
rng = np.random.default_rng(88)
earthquakes = np.unique(event_id)
rng.shuffle(earthquakes)

is_train = np.isin(event_id, earthquakes[:int(0.7 * len(earthquakes))])
is_test = ~is_train
up_train = up[is_train]
up_test = up[is_test]

always_up = np.full(len(up_test), True)

print("training on", is_train.sum(), "recordings from",
      int(0.7 * len(earthquakes)), "earthquakes")
print("held out:  ", len(up_test), "recordings from the other", len(earthquakes) - int(0.7 * len(earthquakes)))
print("always say up:", round(accuracy_score(up_test, always_up), 3))

That last number is the one every method in this notebook has to beat. `up` is slightly the more
common letter, so a rule that ignores the waveform completely and answers *up* every time is
already right about half the time.

**Baseline:** Write the dumbest rule you can, first. Any model that cannot beat it is decoration.

Now the rule the physics suggests, in one line: take the average of the 10 samples of
background just before the pick, and ask whether the trace has gone above it or below it a moment
later. `cut` puts those 10 lead-in samples at the front of every window, so
`piece[:, :LEAD_IN]` is the background and `piece[:, LEAD_IN]` is the sample at the pick itself.
The line below reads 2 samples further on, because an analyst's pick usually lands a
sample or two before the ground has actually started moving.

In [ ]:
piece = cut(0.1)
background = piece[:, :LEAD_IN].mean(axis=1)
first_swing = piece[:, LEAD_IN + 2] - background
rule_says_up = first_swing[is_test] > 0

print("one line of arithmetic:", round(accuracy_score(up_test, rule_says_up), 3),
      "on", len(up_test), "held-out recordings")
print("always say up:        ", round(accuracy_score(up_test, always_up), 3))

One subtraction and a comparison, no training, no fitting, nothing learned from the
1,629 training recordings at all — and it is right about four times in five. That is
the number the rest of this notebook is measured against, and it is deliberately the *first*
thing here rather than an afterthought.

`rule_says_up` is one True/False per held-out recording, not a score. Keep every method in that
shape: a method that hands back its accuracy can only ever be compared, while one that hands back
its answers can also be asked *which* ones it got wrong, which is where this project ends up.

### ✏️ Your turn 1

Three things about that line are choices rather than physics, and none of them was justified
above:

- **which sample** it reads — `LEAD_IN + 2` was asserted, not argued;
- **how much background** it averages — `LEAD_IN` is 10 samples, and could be 3 or 100;
- **one sample or several** — `piece[:, LEAD_IN + 2]` reads one number, but the mean of
  the first few samples after the pick is just as much "one line".

Try each. Score every version with `accuracy_score(up_test, ...)` so the numbers sit on the same
held-out recordings and can be compared, and print each one as you go.

Then print one more line answering it in a sentence, on your own numbers: **what is the best
score one line of arithmetic reaches on this data, and which of the three choices moved it
most?**

In [ ]:
# ← your answer here



## How much waveform does the network need?

A network is not restricted to one sample. Hand it the whole window and it can use anything in
there: the shape of the swing, how fast it decays, how the coda rings, how far away the S wave
is. All of that is real information about the earthquake.

The question this track is built on is whether any of it is information about the *polarity*.
`cut(seconds)` is the one knob — it is the only thing that changes between the runs below, and
every window starts at the same place and differs only in how far past the pick it reaches.

### Predict before you run

You are about to train the same network on windows from a tenth of a second to ten seconds long.
Which one do you think will score highest on the held-out recordings? Change `my_guess_seconds`
and run the cell — you will find out two questions from now, and a wrong guess you committed to
is worth more than a right answer you were shown.

In [ ]:
my_guess_seconds = 2

print("I think", my_guess_seconds, "seconds of waveform after the pick will score highest")

### ✏️ Your turn 2

Write **one** function, and give it exactly this shape:

```python
def network_says_up(seconds):
    """Train a network on `seconds` of waveform after the pick; one True/False per held-out
    recording."""
```

The recipe, in words:

1. `window = cut(seconds)` gives one row per recording. PyTorch wants a channel dimension on a
   1-D convolution, so hand it `torch.tensor(window).unsqueeze(1)` — **`unsqueeze(1)`** inserts a
   length-1 dimension in the middle, turning *(recordings, samples)* into *(recordings, 1,
   samples)*. Split that with `is_train` and `is_test`, the same masks the labels use.
2. The thing to learn is a sign, so make the target a sign: `+1.0` where the polarity is up and
   `-1.0` where it is down. `np.where(up_train, 1.0, -1.0)` builds it, and
   `.reshape(-1, 1)` gives it the shape the network's single output has.
3. The network: two `nn.Conv1d` layers with `nn.ReLU` after each, then **`nn.Flatten()`** — which
   lays the convolution's output out as one long row per recording — then one `nn.Linear` down to
   a single number. Put `torch.manual_seed(0)` before you build it so a re-run repeats.
4. Train it the way you trained the picker: `torch.optim.Adam`, `nn.MSELoss`, batches of 32, a
   fixed number of epochs, `torch.randperm` to shuffle the order each time.
5. Hand back `output > 0` — one True/False per held-out recording, **not** the fraction it got
   right. A function that returns the fraction can only be compared with another fraction; one
   that returns the answers can be asked which recordings it got wrong, which is what the last
   two sections of this notebook do.

Then run it on the shortest window there is — a tenth of a second — and score it against
`up_test`. This is a slow cell; nothing more prints until it has finished.

Print one line answering it in a sentence, on your own two numbers: **did the network beat the
one line you sharpened in Your turn 1, and by how much?**

In [ ]:
# ← your answer here



One window is one point. The fork this project turns on is the whole curve.

### ✏️ Your turn 3

Run `network_says_up` at every window length in `[0.1, 0.25, 0.5, 1, 2, 5, 10]` seconds and score each one on
`up_test`. This is the slow cell in the notebook — it trains the same network seven times over —
so print each score as it arrives rather than at the end.

Draw the result: held-out accuracy against window length, with your best one-line rule and the
always-say-up baseline marked as horizontal lines so all three are readable together. A log
x-axis (`plt.xscale("log")`) spaces the windows evenly.

Then print one more line answering it in a sentence, on your own curve: **which window wins, and
what does the shape of the curve say about what the network is doing with the extra seconds?**

In [ ]:
# ← your answer here



## Did the network earn its place?

You now have three numbers on the same 719 held-out recordings: a rule that ignores the
data, a rule that reads one number out of it, and a network that chose its own weights from
1,629 training recordings and took far longer than either to produce.

**Overfitting:** A curve that memorises the data you gave it fails on the data you did not.

### ✏️ Your turn 4

Two or three paragraphs, quoting **your own three numbers** — the baseline, your best one line,
and the network at its best window.

1. Did the network earn its place here? Say what it would have to score before you would use it
   instead of the one line, and why that threshold and not a smaller one. Remember that the held
   out set is 719 recordings, so one percentage point is a countable number of them —
   work out how many, and say it.
2. Whichever of the two is ahead: how big would the gap have to be before you believed it was
   real rather than an accident of which earthquakes landed in the held-out set? Name the thing
   you would run to find out, and what its answer would look like either way.

*(Double-click this cell and replace this line with your answer.)*

Both methods hand back one True/False per held-out recording, which means they can be compared
recording by recording rather than only in aggregate. Two methods can reach the same score by
being right about the same recordings, or by being right about different ones — and those are
completely different findings.

### ✏️ Your turn 5

Take your best one line from *Your turn 1* and the network's answers at its best window, and count
the four cases: both right, both wrong, only the line right, only the network right. Print all
four, and print how often the two methods simply agree with each other regardless of who is
right.

Then draw two or three of the recordings where they disagree — the same figure as the one at the
top of this notebook, one trace at a time — and look at them. If the first ones you draw all look
like each other, that is a finding rather than a bug: count how many of the disagreements look
that way, and keep drawing until you have seen one that does not.

Print one more line answering it in a sentence, on your own four counts: **is the network doing
something different from the one line, or the same thing slightly better?**

In [ ]:
# ← your answer here



## Is what is left the model, or the labels?

None of the numbers you have collected is 1.0, and what the missing part is made of is a different
kind of question from any asked so far. Either every method here is too weak and a better one
would keep climbing — or the recordings being got wrong do not have a readable answer in them,
and the analyst who wrote the letter was guessing too.

Those can be told apart, and the tool is the size of the first swing against the background noise
before it. A swing ten times the background is unambiguous; a swing the size of the background is
a coin toss whoever is reading it.

### ✏️ Your turn 6

Measure how clear each first motion is: the size of `first_swing` against the typical size of the
10 background samples before the pick. `piece[:, :LEAD_IN].std(axis=1)` gives you that
background size, one number per recording.

Then, over **all** 2,348 recordings rather than only the held-out ones — this is a
question about the labels, not about a model, so nothing is being fitted and nothing is being
scored — split them into four groups by that clarity and, in each group, work out how often the
sign of the waveform agrees with the letter the analyst wrote. Plot it.

Print one more line answering it in a sentence, on your own four numbers: **on the clearest
arrivals, how often does the waveform agree with the analyst — and does what is left over look
like the model's problem or the labels'?**

In [ ]:
# ← your answer here



## The question, answered

The ground's first move is written in the three or four samples immediately after the P pick, and
one line of arithmetic reads it correctly on 80.3% of the 719 held-out
recordings against 52.0% for a rule that never looks at the waveform — so yes, a
machine can tell, and it does not need to be much of a machine. What this notebook has deliberately not told
you is where your own network landed on that scale, and that comparison, not the network, is the
project.

## What track T8 leans on

**The question.** Which way did the ground first move, and can a machine tell?

Nothing here is new. These are the weeks to look back at while you work, and the wording is the course's own.

Two calls this track needs are not in any of those tables, because no week has wanted them yet, and both are named where you first need them: `tensor.unsqueeze(1)`, which adds the single-channel dimension a `nn.Conv1d` expects, and `nn.Flatten()`, which lays a convolution's output out flat so an `nn.Linear` can read it.

### The ideas, in plain words

| Idea | Means |
|---|---|
| **Baseline** | Write the dumbest rule you can, first. Any model that cannot beat it is decoration. |
| **Train/test split** | Hide some data from yourself, then check. |
| **Overfitting** | A curve that memorises the data you gave it fails on the data you did not. |
| **1-D CNN** | Slide a small pattern-detector along the signal. |
| **Loss** | The one number the network is trying to make small. |
| **Epoch** | One pass through all of the training data. |

### Code you will reach back for

| Function | What it does |
|---|---|
| `accuracy_score(y, pred)` | the fraction it got right — useless when one class is 98% of the data |
| `np.unique(a)` | each different value in an array, once |
| `np.isin(a, values)` | a mask marking which items appear in a list of allowed values |
| `torch.tensor(array)` | hand an array to PyTorch so it can be learned from |
| `torch.manual_seed(n)` | fix the random start, so a training run repeats |
| `nn.Sequential(layers)` | a model that runs the layers you hand it, in order |
| `nn.Conv1d(in, out, width)` | a row of pattern-detectors slid along the signal, with weights the training chooses |
| `nn.ReLU()` | the activation — the bend that makes a stack more than one straight line |
| `nn.Linear(in, out)` | one layer of weighted sums — a row of perceptrons |
| `nn.MSELoss()` | the loss: the average squared miss, the same one that fits a straight line |
| `torch.optim.Adam(model.parameters(), lr=)` | the thing that rolls the weights downhill |
| `loss.backward() / optimiser.step() / optimiser.zero_grad()` | work out which way each weight should move, move them, then clear the slate |
| `torch.randperm(n)` | a shuffled order, for going through the data differently each epoch |
| `tensor.detach().numpy()` | take the numbers back out of PyTorch |

## What your project must contain

Five sections, empty below, required of **every** EPS 88 project regardless of track. They are
headed here so the shape of a good answer is visible while you work. Fill them in as you go; they
are not a write-up you do at the end.

### ✏️ 1 · A one-sentence answer

Your claim and its uncertainty, in one sentence, at the top of your report. If you cannot put a
number and a range in it, you do not have a result yet.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 2 · The trivial baseline

Before any model, state the dumbest answer to your question and what it gives. Every later number
is reported against it.

This track hands you two, on purpose and in that order: a rule that never reads the waveform, and
a rule that reads one number out of it. Say what each of them gives on your split, and say what
your network bought you over the better of the two — in accuracy, and in what it cost to run.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 3 · Split by structure

Earth data are correlated in space and in time, so a train/test split has to follow the structure
that is really there — never a random cut across rows.

The split here is by earthquake, and the setup cell made that choice for you. Say why it is the
right one on this data, what a random split would have leaked, and — if you have the patience —
what happens to your numbers when you make the split by *station* instead.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 4 · What I got wrong

What failed, and what you believed before it failed. Honest failure is graded; a faked success is
not. Your *Predict before you run* guess belongs here if it was wrong.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 5 · AI disclosure

Which tool, what you asked it, what you changed in what it gave you, and how you checked that the
result was true.

*(Double-click this cell and replace this line with your answer.)*

## The open question

> **Is the remaining gap model capacity, or is it the labels?**

Nobody grading this knows the answer, and neither does the literature. Everything above is the
scaffolding; this is the project.

Here is what is actually established, and it is less than it looks. That the polarity lives in
the first few samples is settled — the sweep shows it, and one subtraction reads it. What is
**not** settled is why every method here stops in the same place, and the two candidate
explanations make different predictions that this dataset can be made to distinguish.

The question as it is written above offers two answers, and this notebook has already turned up
evidence for a third: 186 of the 2,348 recordings are a single flat line
at the file's own ±10 cut-off for their whole length, with no arrival in them at
all. That is 7.9% of the data on which no method can be better
than a coin toss, and it is neither the model's fault nor the analyst's. How much of the ceiling
is that, how those recordings got into the file, and whether an honest project should drop them —
saying so, and reporting both numbers — is the first thing to settle, and nothing above settles
it.

Four more directions, none of them worked out here:

1. **Score the labels, not the models.** Section 4 measured agreement between the waveform and
   the analyst as a function of clarity. Turn that around: for the recordings where the two
   disagree *at high clarity*, one of the two is simply wrong, and it can be decided by eye. Look
   at twenty of them yourself and count. If most are analyst errors, the ceiling is the labels
   and no model will pass it; if most are the rule's, the rule is worse than it looks.
2. **Ask the earthquake, not the recording.** Every earthquake here is recorded at many stations,
   and the true first motions across those stations are not independent — they are set by one
   fault orientation and the station's position on it. A method that reads all the recordings of
   one earthquake together has information no single-trace method has. Nothing in this notebook
   uses it.
3. **Watch the network memorise.** The sweep scores only the held-out set. Score the *training*
   set at every window as well, and the two curves together say whether the long-window collapse
   is a failure to learn or a success at memorising. That is one extra line inside
   `network_says_up`, and it turns a result into a mechanism.
4. **Give it something other than raw samples.** Everything above hands the model the waveform as
   it is. A filtered version, the derivative, or the trace divided by its own first-swing size are
   all the same information rearranged — and if any of them moves the score, what moved it was the
   representation and not the model.

And one that is bigger than a semester: polarity is not wanted for its own sake, it is wanted
because enough polarities determine a fault plane. A method that is right 80% of the
time on single recordings feeds errors into that inversion at a rate nobody in this notebook has
measured. How accurate does a per-recording polarity have to be before the mechanism it produces
is usable — and is 80% already enough, given how many stations a real earthquake is
recorded at? If the answer is that 80% is plenty, then the whole question of beating
the one line was the wrong one to ask, and saying so is a result.

### ✏️ Your turn 7 — the first move

Before you close this notebook: in a few sentences, name the **one** measurement you would make
first, say what it would show if the ceiling is the labels, what it would show if it is the
models, and name the number that would change your mind. Then make it, in the cell below the
prose.

*(Double-click this cell and replace this line with your answer.)*

In [ ]:
# ← your answer here

